# Smart Waste: training dan evaluasi
Pilih Runtime > Change runtime type > T4 GPU. Notebook ini berisi source persiapan dan training agar tidak membutuhkan repository yang sudah dipublikasikan.

Dataset: RealWaste, Single, Iranmanesh & Raad (2023), UCI, https://doi.org/10.24432/C5SS4G. Tinjau lisensi sumber sebelum menjalankan atau mendistribusikan dataset. Target testing 80% harus dibuktikan; notebook tidak menjanjikan hasil.

In [ ]:
%pip install -q tensorflow==2.20.0 keras==3.12.0 numpy==2.2.6 Pillow==12.1.1 scikit-learn==1.7.2 matplotlib==3.10.7
import sys
from importlib.metadata import version
loaded_numpy = sys.modules.get('numpy')
if loaded_numpy is not None and loaded_numpy.__version__ != version('numpy'):
    raise RuntimeError('NumPy kernel masih versi lama. Pilih Runtime > Restart session and run all. Instalasi yang sudah selesai tetap tersimpan.')

In [ ]:
import os
os.chdir('/content')
import tensorflow as tf
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus)
if not gpus:
    raise RuntimeError('GPU belum terdeteksi. Pilih Runtime > Change runtime type > T4 GPU lalu hubungkan kembali.')

## Persiapan dataset
Hanya Food Organics, Vegetation, Paper, dan Plastic digunakan. Pemeriksaan duplikasi dilakukan sebelum split. Simpan manifest asli; jangan ubah test set berdasarkan hasil model.

In [ ]:
"""Download RealWaste and create a reproducible, duplicate-grouped split."""
import argparse
import csv
import hashlib
import io
import json
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

SOURCE = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"
MAPPING = {"Food Organics": "organic", "Vegetation": "organic", "Paper": "paper", "Plastic": "plastic"}
LABELS = ["organic", "paper", "plastic"]


def canonical_image(raw):
    with Image.open(io.BytesIO(raw)) as image:
        image.load()
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((2048, 2048))
        output = io.BytesIO()
        image.save(output, "JPEG", quality=88)
        return output.getvalue()


def difference_hash(raw):
    with Image.open(io.BytesIO(raw)) as image:
        pixels = np.asarray(image.convert("L").resize((9, 8), Image.Resampling.BILINEAR))
    return int("".join("1" if v else "0" for v in (pixels[:, 1:] > pixels[:, :-1]).flat), 2)


def prepare(root="data"):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    archive_path = root / "realwaste.zip"
    if not archive_path.exists():
        partial = archive_path.with_suffix(".partial")
        urllib.request.urlretrieve(SOURCE, partial)
        partial.replace(archive_path)
    rows, rejected = [], []
    images = root / "images"
    images.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in sorted(archive.namelist()):
            path = Path(member)
            if path.parent.name not in MAPPING or path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            try:
                raw = archive.read(member)
                image = canonical_image(raw)
                checksum = hashlib.sha256(raw).hexdigest()
                label = MAPPING[path.parent.name]
                target = images / f"{checksum}.jpg"
                target.write_bytes(image)
                rows.append({"path": target.relative_to(root).as_posix(), "source_path": member,
                             "category": label, "sha256": checksum, "dhash": difference_hash(image)})
            except (OSError, ValueError):
                rejected.append({"source_path": member, "reason": "decode_failed"})
    if len(rows) < 150:
        raise RuntimeError("Dataset kurang dari 150 gambar; periksa unduhan dan struktur sumber")
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, item in enumerate(rows):
        for j in range(i):
            other = rows[j]
            if item["sha256"] == other["sha256"] or (item["dhash"] ^ other["dhash"]).bit_count() <= 4:
                parent[find(i)] = find(j)
    groups = {}
    for i, item in enumerate(rows):
        groups.setdefault(find(i), []).append(item)
    clean = []
    for group in groups.values():
        if len({r["category"] for r in group}) > 1:
            rejected.extend({"source_path": r["source_path"], "reason": "cross_label_duplicate_group"} for r in group)
            continue
        # Retain one representative; near-duplicates cannot cross the split.
        representative = group[0]
        representative["group"] = representative["sha256"]
        clean.append(representative)
        rejected.extend({"source_path": r["source_path"], "reason": "duplicate"} for r in group[1:])
    counts = Counter(r["category"] for r in clean)
    if any(counts[label] < 50 for label in LABELS):
        raise RuntimeError(f"Butuh minimal 50 gambar unik per kelas: {counts}")
    train, held_out = train_test_split(clean, test_size=0.30, random_state=42,
                                      stratify=[r["category"] for r in clean])
    validation, test = train_test_split(held_out, test_size=0.50, random_state=42,
                                       stratify=[r["category"] for r in held_out])
    for split, items in (("train", train), ("validation", validation), ("test", test)):
        for item in items:
            item["split"] = split
    fields = ["path", "source_path", "category", "sha256", "group", "split"]
    with (root / "manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(clean)
    with archive_path.open("rb") as source_stream:
        archive_sha256 = hashlib.file_digest(source_stream, "sha256").hexdigest()
    summary = {"source": SOURCE, "source_sha256": archive_sha256,
               "seed": 42, "mapping": MAPPING, "unique_images": len(clean), "counts": dict(counts),
               "split_counts": dict(Counter(f'{r["split"]}/{r["category"]}' for r in clean)),
               "duplicate_policy": "SHA256 or dHash Hamming <=4; conflicting-label groups excluded", "rejected": rejected}
    (root / "dataset_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in summary.items() if k != "rejected"}, indent=2))
    return root / "manifest.csv"




In [ ]:
prepare('/content/data')

## Training, pemilihan checkpoint, dan evaluasi final
MobileNetV2 memakai bobot ImageNet. Model yang dipilih berdasarkan validation loss baru kemudian diuji. Jangan mengubah hyperparameter menggunakan test set.

In [ ]:
%%writefile /content/pixels.py
"""Image input contract: RGB, PIL bilinear 224x224, float32 in 0..255."""
import numpy as np
from PIL import Image


def load_pixels(path):
    if isinstance(path, bytes):
        path = path.decode()
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB").resize((224, 224), Image.Resampling.BILINEAR), dtype=np.float32)


In [ ]:
"""Train on training/validation only, then evaluate the selected model once on test."""
import argparse
import csv
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from pixels import load_pixels
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

LABELS = ["organic", "paper", "plastic"]


def train(data="data", output="artifacts", head_epochs=15, fine_epochs=10):
    tf.keras.utils.set_random_seed(42)
    tf.config.experimental.enable_op_determinism()
    data, output = Path(data), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    rows = list(csv.DictReader((data / "manifest.csv").open(encoding="utf-8")))
    for key in ("sha256", "group"):
        sets = [{r[key] for r in rows if r["split"] == split} for split in ("train", "validation", "test")]
        assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]), "Data leakage"

    def dataset(split, shuffle=False):
        selected = [r for r in rows if r["split"] == split]
        paths = [str(data / r["path"]) for r in selected]
        labels = np.array([LABELS.index(r["category"]) for r in selected], dtype=np.int32)

        def generate():
            for path, label in zip(paths, labels):
                yield load_pixels(path), label

        ds = tf.data.Dataset.from_generator(generate, output_signature=(tf.TensorSpec((224, 224, 3), tf.float32), tf.TensorSpec((), tf.int32)))
        if shuffle:
            ds = ds.shuffle(len(selected), seed=42, reshuffle_each_iteration=True)
        return ds.batch(32).prefetch(tf.data.AUTOTUNE), labels

    training, train_labels = dataset("train", True)
    validation, _ = dataset("validation")
    classes = np.arange(3)
    weights = dict(zip(classes.tolist(), compute_class_weight("balanced", classes=classes, y=train_labels).tolist()))
    base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights="imagenet")
    base.trainable = False
    inputs = tf.keras.Input((224, 224, 3), name="rgb_0_255")
    x = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal", seed=42), tf.keras.layers.RandomRotation(.1, seed=42)], name="augmentation")(inputs)
    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocessing")(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(.2, seed=42)(x)
    model = tf.keras.Model(inputs, tf.keras.layers.Dense(3, activation="softmax", name="probabilities")(x))

    def callbacks(name):
        return [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
                tf.keras.callbacks.ModelCheckpoint(str(output / name), monitor="val_loss", save_best_only=True),
                tf.keras.callbacks.TerminateOnNaN()]

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    head = model.fit(training, validation_data=validation, epochs=head_epochs, class_weight=weights, callbacks=callbacks("head.keras"))
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    fine = model.fit(training, validation_data=validation, epochs=fine_epochs, class_weight=weights, callbacks=callbacks("fine.keras"))
    chosen = "fine.keras" if min(fine.history["val_loss"]) < min(head.history["val_loss"]) else "head.keras"
    model = tf.keras.models.load_model(output / chosen)
    model.save(output / "model.keras")
    # Testing is opened only after the checkpoint is fixed using validation loss.
    testing, actual = dataset("test")
    scores = model.predict(testing)
    predicted = scores.argmax(axis=1)
    accuracy = float(accuracy_score(actual, predicted))
    matrix = confusion_matrix(actual, predicted, labels=classes)
    report = classification_report(actual, predicted, labels=classes, target_names=LABELS, output_dict=True, zero_division=0)
    evaluation = {"test_accuracy": accuracy, "target_accuracy": .8, "meets_target": accuracy >= .8,
                  "test_count": len(actual), "confusion_matrix": matrix.tolist(), "classification_report": report,
                  "selected_checkpoint": chosen, "selection_metric": "validation_loss"}
    (output / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
    (output / "classification_report.txt").write_text(classification_report(actual, predicted, labels=classes, target_names=LABELS, zero_division=0), encoding="utf-8")
    ConfusionMatrixDisplay(matrix, display_labels=LABELS).plot(cmap="Greens")
    plt.tight_layout(); plt.savefig(output / "confusion_matrix.png", dpi=160); plt.close()
    histories = {key: head.history[key] + fine.history.get(key, []) for key in head.history}
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, metric in zip(axes, ("accuracy", "loss")):
        axis.plot(histories[metric], label="train"); axis.plot(histories[f"val_{metric}"], label="validation")
        axis.set_title(metric); axis.set_xlabel("epoch"); axis.legend()
    fig.tight_layout(); fig.savefig(output / "training_curves.png", dpi=160); plt.close(fig)
    metadata = {"version": datetime.now(timezone.utc).strftime("mobilenetv2-%Y%m%dT%H%M%SZ"),
                "labels": LABELS, "input_size": [224, 224], "preprocessing": "embedded_mobilenet_v2",
                "resize": "PIL_BILINEAR", "tensorflow": tf.__version__, "seed": 42,
                "sha256": hashlib.file_digest((output / "model.keras").open("rb"), "sha256").hexdigest()}
    (output / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    np.savez_compressed(output / "test_predictions.npz", actual=actual, probabilities=scores)
    for name in ("manifest.csv", "dataset_summary.json"):
        shutil.copy2(data / name, output / name)
    (output / "training_history.json").write_text(json.dumps(histories, indent=2), encoding="utf-8")
    print(json.dumps(evaluation, indent=2))
    print("PASS: target tercapai" if accuracy >= .8 else "BELUM LULUS: laporkan hasil ini secara jujur dan perbaiki eksperimen berdasarkan validation")
    return output




In [ ]:
train('/content/data', '/content/artifacts')

In [ ]:
from IPython.display import display, Image as DisplayImage
display(DisplayImage('/content/artifacts/confusion_matrix.png'))
display(DisplayImage('/content/artifacts/training_curves.png'))
print(open('/content/artifacts/classification_report.txt').read())

## Unduh hasil
Arsip berisi model.keras, metadata.json, evaluasi asli, grafik, manifest, dan prediksi testing. Berikan arsip ini untuk integrasi API. Bila ingin menyimpan dataset untuk pengumpulan, unduh realwaste.zip terpisah dan cantumkan atribusi/lisensinya.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/smart-waste-artifacts', 'zip', '/content/artifacts')
files.download('/content/smart-waste-artifacts.zip')